<div style="background:linear-gradient(90deg,#0366d6,#28a745); color:white; padding:20px 32px; border-radius:8px; width:97%;">

# 🔬 `agent.invoke()` 完整执行轨迹解读

</div>

本笔记解读 [2_a_🔥_files.ipynb](./2_a_🔥_files.ipynb) 里这段代码的执行:

```python
result = agent.invoke(
    {
        "messages": [
            {"role": "user",
             "content": "Give me an overview of Model Context Protocol (MCP)."}
        ],
        "files": {},
    }
)
format_messages(result["messages"])
```

对应的 raw 输出在 [`messages.txt`](./messages.txt) —— **8 条消息**完整展示了 Manus 风格的 **"orient → save → search → read-back → answer"** 五步循环。

## 🏗️ Graph 结构回忆

```
START → agent ──┬──► tools ──► agent   (回环)
                └──► END                (条件边:看 AIMessage.tool_calls 是否为空)
```

- 🤖 **`agent` 节点** = 调用 LLM,产出一个 `AIMessage`
- 🔧 **`tools` 节点** = `ToolNode`,并发执行 AIMessage 里的所有 `tool_calls`
- 🚦 **退出条件** = LLM 产生不带 `tool_calls` 的 AIMessage

## 🗺️ 8 条消息总览

```
[0] Human:    "Give me an overview of MCP"
[1] AI    ── tool_calls=[ls(), write_file(mcp_request.txt), web_search(MCP)]   ← 第1轮:并行 3 个
[2] Tool   ← ls()         返回 []
[3] Tool   ← write_file   返回 "Updated file mcp_request.txt"
[4] Tool   ← web_search   返回 mock MCP 描述
[5] AI    ── tool_calls=[read_file(mcp_request.txt)]                          ← 第2轮:读回文件
[6] Tool   ← read_file 返回带行号的文件内容
[7] AI    ── content="## Overview of MCP ..."  tool_calls=[]                  ← 最终答案,END
```

**3 轮 LLM 调用、5 次工具执行(3 + 1 + 1)、1 次最终回答**。

## 📍 Message [0] — `HumanMessage`

```python
HumanMessage(
    content='Give me an overview of Model Context Protocol (MCP).',
    id='68c74733-a605-...'
)
```

用户原始请求,被 LangChain 自动包成 `HumanMessage`,加一个 UUID。

## 📍 Message [1] — AIMessage 含 **3 个并行 tool_calls**

```python
AIMessage(
    content=[
        {'text': "I'll help you get an overview ...", 'type': 'text'},
        # 三个 tool_use
    ],
    tool_calls=[
        {'name': 'ls',         'args': {},                                       'id': 'toolu_01Ut...'},
        {'name': 'write_file', 'args': {'file_path': 'mcp_request.txt', ...},    'id': 'toolu_01Xq...'},
        {'name': 'web_search', 'args': {'query': 'Model Context Protocol MCP'}, 'id': 'toolu_01Cg...'},
    ],
    response_metadata={'stop_reason': 'tool_use', ...},
    usage_metadata={'input_tokens': 1305, 'output_tokens': 214},
)
```

### 🎯 LLM 这里干了什么?

完全照搬 `FILE_USAGE_INSTRUCTIONS` 的前两步:

| 步骤 | 调用 | 原因 |
|---|---|---|
| 1️⃣ **Orient** | `ls()` | 先看现有文件 |
| 2️⃣ **Save** | `write_file("mcp_request.txt", ...)` | 保存用户请求 + 计划要 cover 的子话题 |
| 3️⃣ **同时启动**| `web_search("...")` | 反正搜索不依赖前两步,顺手并行了 |

> 🔑 **关键观察**:这 3 个 tool_call 之间**没有依赖**,所以 LLM 把它们打包进**同一个 AIMessage**,`ToolNode` 会**并发执行**。这是 ReAct + 并行调用机制的体现。

`input_tokens=1305`:此时上下文 = system_prompt + user_message + 4 个工具的 schema。

<div style="background:#fff3b0; padding:12px 24px; border-left:4px solid #f0ad4e; border-radius:4px; width:97%;">

### 🧩 插问:`mcp_request.txt` 里那些 \"Topics to cover\" 是谁加的?

</div>

注意 Message [1] 里 `write_file` 的 `args.content` 是这样一坨:

```
User Request: Give me an overview of Model Context Protocol (MCP)

Topics to cover:
- What is MCP?
- Purpose and goals
- Key features
- How it works
- Use cases
- Who developed it
```

**用户的原始请求只有 1 句话** —— 那 6 项 \"Topics to cover\" 是哪冒出来的?

---

#### 🎯 你可能有的误解

- "**代码里哪一行加的 Topics?我找不到!**"
- "肯定是 LangChain 在某处偷偷加了吧?"
- "`write_file` 工具内部对 content 做了加工?"

**都不是。** Topics 是 LLM **当场打字打出来的** —— 跟它生成最终那段 \"## Overview of MCP...\" 是**完全一样的机制**。

---

#### 💡 核心认知:LLM 只做一件事 —— 生成文本

LLM 没有什么 \"工具调用模式\"。它产出的**所有东西**,无论看起来是自然语言还是 tool call,**本质都是 LLM 一个 token 一个 token 吐出来的字符串**。

举个简化的例子。

**LLM 看到的输入**:

```
System: 用 write_file() 保存用户请求,以便之后参考
User:   Give me an overview of MCP
```

**LLM 生成的输出**(全是它一个 token 一个 token 打出来的):

```
I'll help organize my workspace.

<tool_use>
{
  \"name\": \"write_file\",
  \"args\": {
    \"file_path\": \"mcp_request.txt\",
    \"content\": \"User Request: Give me an overview...
                  Topics to cover:
                  - What is MCP?
                  - Purpose and goals
                  ...\"           ← 👈 这一整段字符串,就是 LLM 自己写的
  }
}
</tool_use>
```

**`content` 里那一大坨字符串 = LLM 一个 token 一个 token 写出来的**,跟前面那句 \"I'll help...\" 完全同一回事。

LangChain 拿到这串 token 之后:

1. 解析 `<tool_use>` 块
2. 把它包成 `tool_calls=[{...}]`
3. 调用真实的 `write_file` 函数,把 `content` **原样**传进去

**LangChain 没\"加任何内容\"** —— 纯搬运。

---

#### 🧠 最直观的类比:让朋友帮你记笔记

假设你对朋友说:

> \"把我今天的待办记到本子上,我等下要翻。\"

然后你只说了一句:

> \"买牛奶。\"

朋友会怎么写?

| 朋友类型 | 笔记本上写啥 |
|---|---|
| 🤖 **机械型** | \"买牛奶。\" |
| 🧠 **聪明型** | **\"TODO**<br>- 买牛奶(顺便看看面包打折没)<br>- 我可能还需要鸡蛋?\" |

聪明的朋友**不会机械转录**,他**会自己加结构、加联想到的相关项**,因为他**心里有 \"等下要翻\" 这件事**,知道笔记越有用越好。

**LLM 就是那个聪明型朋友**。它收到 \"存请求\" 指令,但它的\"大脑\"输出的不只是字面请求 —— 它**主动加了任务大纲**,因为它知道接下来还要 `read_file` 读回来,**给未来的自己写笔记**。

---

#### 🔁 跟前面 calculator 例子是同一回事

回忆 0_a 里的 `calculator(operation=\"add\", a=15, b=23)`:

| 参数 | 哪来的? |
|---|---|
| `operation=\"add\"` | LLM **自己想出来**的(基于 \"+\" 推理)|
| `a=15` | LLM **从用户问题里读**出来的 |
| `b=23` | 同上 |

这些参数值**没有任何一个是 \"代码逻辑提取的\"**,**全是 LLM 在那一瞬间生成的**。

`write_file(file_path=\"mcp_request.txt\", content=\"...\")` 也完全一样:

| 参数 | 哪来的? |
|---|---|
| `file_path=\"mcp_request.txt\"` | LLM **自己起的名字**(下次跑可能叫 `request.txt`)|
| `content=\"User Request:...Topics:...\"` | LLM **自己写的内容**(大纲是它\"觉得有用\"加的)|

---

#### 🧪 怎么验证这件事?

**换个用户请求,topics 会完全不一样**。

如果用户问 \"Give me a recipe for tomato pasta\",LLM 会写出:

```json
{
  \"file_path\": \"recipe_request.txt\",   ← 名字变了
  \"content\": \"User Request: tomato pasta recipe\\n\\nDetails to cover:\\n- Ingredients\\n- Step-by-step instructions\\n- Cooking time\\n- Difficulty level\\n...\"
}                                          ↑ 大纲完全变了
```

→ 不是固定模板!**Topics 是 LLM 即兴生成的**,跟用户问什么直接相关。

---

#### ✨ 一句话核心

<div style=\"background:#e7f7e9; padding:10px 24px; border-left:4px solid #28a745; border-radius:4px; width:97%;\">

> 🔑 **Tool call 里的所有参数(包括 `content` 那一大坨字符串)= LLM 当场打字打出来的**。
>
> 不是代码提取、不是模板填充、不是框架加工。**LLM 是唯一作者**。
>
> 就像你让 LLM \"写首诗\",它会写;让它\"产生一个 `write_file` 调用\",它就**连参数一起写**。**写诗和写参数,对 LLM 来说是同一件事**。

</div>

---

#### 🎯 验证:这套规划在 Message [7] 真的兑现了

回看最终答案的小节标题 —— `### What is MCP?` / `### Developer` / `### Purpose and Goals` / `### How It Works` / `### Key Benefits` / `### Use Cases` —— **6 项 topics 全部命中**。

证明:**LLM 写下计划 → 读回计划 → 按计划答题**这个闭环真的形成了。如果 [5] 没读 file,这次执行很可能漏掉 \"Who developed it\" 或 \"Use cases\" 之类的子话题。

## 📍 Message [2][3][4] — 3 条并发 ToolMessage

```python
ToolMessage(content=[],                              name='ls',         tool_call_id='toolu_01Ut...')  # 空文件系统
ToolMessage(content='Updated file mcp_request.txt',  name='write_file', tool_call_id='toolu_01Xq...')
ToolMessage(content='The Model Context Protocol ...',name='web_search', tool_call_id='toolu_01Cg...')
```

### 🔍 三个 ToolMessage 对应关系

| 工具 | 返回值 | state 变化 |
|---|---|---|
| 📋 `ls()` | `[]` 空数组 | 无变化(初始 `files={}`) |
| ✍️ `write_file` | 一句确认 | **`state.files` 新增 `mcp_request.txt`** 🎯 |
| 🔍 `web_search` | mock 的 MCP 介绍 | 无(纯返回文本)|

> 🪪 三条 ToolMessage 的 `tool_call_id` **必须**精确匹配 [1] 里 `tool_calls` 各自的 `id`。这是 LLM 对应"我发的 3 个调用 → 我收到的 3 个结果"的唯一依据。

### ⚙️ `write_file` 触发的 state 更新

回忆 `write_file` 内部:

```python
return Command(update={
    "files": {"mcp_request.txt": "User Request: ..."},  # → 经 file_reducer 并入 state
    "messages": [ToolMessage("Updated file...", tool_call_id=...)],
})
```

经 `file_reducer` 合并后,`state.files` 变成 `{"mcp_request.txt": "User Request: ..."}`。

## 📍 Message [5] — AIMessage 带 `read_file` tool_call

```python
AIMessage(
    content=[
        {'text': 'Now let me read the request file and provide you with a comprehensive overview:', ...},
        # tool_use
    ],
    tool_calls=[{'name': 'read_file',
                 'args': {'file_path': 'mcp_request.txt'},
                 'id': 'toolu_01LV...'}],
    usage_metadata={'input_tokens': 1720, 'output_tokens': 76},
)
```

### 🎯 这一步是 FILE_USAGE_INSTRUCTIONS 的精髓

> *"Once you are satisfied with the collected sources, **read the saved file** and use it to ensure that you directly answer the user's question."*

**LLM 在生成最终答案之前,主动读回自己几步前写的文件**。这是 Manus 风格 anti-rot(防跑题)的核心动作:

- 🧠 此时 messages 里塞满了搜索结果,如果不刷新,LLM 容易**只回答搜索结果里有的**,忽略原始请求里要 cover 的子话题("Who developed it"、"Use cases" 等)
- 📖 读回 `mcp_request.txt` → 计划清单**重新出现在 messages 末尾** → LLM 答题前最后看到的就是"该 cover 哪些点"

<div style="background:#fff8e1; padding:10px 24px; border-left:4px solid #ff9800; border-radius:4px; width:97%;">

💡 **这就是为什么文件系统这么强大**:不仅是"存数据",更是**控制 LLM 注意力的开关** —— 想让 LLM 想起什么,就 `read_file` 把那个东西塞回 context 末尾。

</div>

`input_tokens=1720`(+415):整轮重放,包含 [1] 的 AIMessage 和 [2][3][4] 三条 ToolMessage。

## 📍 Message [6] — `read_file` 的 ToolMessage

```python
ToolMessage(
    content='     1\tUser Request: Give me an overview of Model Context Protocol (MCP)\n'
            '     2\t\n'
            '     3\tTopics to cover:\n'
            '     4\t- What is MCP?\n'
            '     5\t- Purpose and goals\n'
            '     6\t- Key features\n'
            '     7\t- How it works\n'
            '     8\t- Use cases\n'
            '     9\t- Who developed it',
    name='read_file',
    tool_call_id='toolu_01LV...'
)
```

### 🔍 注意:**内容带行号**

回忆 `read_file` 的实现:

```python
result_lines.append(f"{i + 1:6d}\t{line_content}")
```

**为什么要加行号?** —— 让 LLM 之后能用 `edit_file(old_line, new_line)` 之类精确引用某一行(类似 `cat -n`)。本节虽没用到 edit,但工具实现已经为此打基础了。

→ 现在 messages 末尾有了 6 条 "Topics to cover"。LLM 注意力被**精准锁定**到原始需求。

## 📍 Message [7] — 最终 AIMessage(END)

```python
AIMessage(
    content='## Overview of Model Context Protocol (MCP)\n\n'
            '### What is MCP?\n...\n'
            '### Developer\n...\n'
            '### Purpose and Goals\n...\n'
            '### How It Works\n...\n'
            '### Key Benefits\n...\n'
            '### Use Cases\n...',
    tool_calls=[],                            # ← 🔑 空,触发 END
    response_metadata={'stop_reason': 'end_turn', ...},
    usage_metadata={'input_tokens': 1886, 'output_tokens': 423},
)
```

### ✅ 退出条件

| 字段 | 之前的 AIMessage | 这个最终 AIMessage |
|---|---|---|
| `content` | 短文本 + `tool_use` blocks | **完整 Markdown 答案** |
| `tool_calls` | 非空列表 | **`[]`** |
| `stop_reason` | `'tool_use'` | **`'end_turn'`** |

LangGraph 路由器看到 `tool_calls=[]` → 跳到 `END`,整个 `agent.invoke()` 返回。

### 🎯 答案恰好覆盖了 `mcp_request.txt` 里列的所有子话题

| 计划里写的子话题 | 最终答案对应小节 |
|---|---|
| What is MCP? | `### What is MCP?` |
| Purpose and goals | `### Purpose and Goals` |
| Key features | `### Key Benefits` |
| How it works | `### How It Works` |
| Use cases | `### Use Cases` |
| Who developed it | `### Developer` |

**6 项全中** —— 证明"先 save 计划、再 read-back"的策略**真的让 LLM 没漏题**。

## ⚙️ `agent.invoke()` 内部到底干了什么?

一步步看:

```
agent.invoke({"messages": [Human], "files": {}})
        │
        ▼
  LangGraph 用 DeepAgentState 初始化 state
        │
        ▼
  START → agent 节点(第 1 轮)
        │
        │   把 state.messages 全部喂给 LLM
        │   LLM 返回 AIMessage[1](3 个 tool_calls)
        │   add_messages reducer:把它追加到 messages
        ▼
  路由器:tool_calls 非空 → tools 节点
        │
        │   ToolNode 并发执行 3 个 tool_call
        │   每个工具返回 ToolMessage(write_file 还带 Command 更新 files)
        │   add_messages 追加 3 条;file_reducer 合并 files
        ▼
  无条件回边 → agent 节点(第 2 轮)
        │
        │   把更新后的 state.messages 全部重新喂给 LLM
        │   LLM 返回 AIMessage[5](1 个 tool_call: read_file)
        ▼
  路由器:tool_calls 非空 → tools 节点
        │
        │   ToolNode 执行 read_file,返回 ToolMessage[6]
        ▼
  无条件回边 → agent 节点(第 3 轮)
        │
        │   把又更新的 messages 重新喂给 LLM
        │   LLM 返回 AIMessage[7](内容长、tool_calls=[])
        ▼
  路由器:tool_calls 为空 → END
        │
        ▼
  agent.invoke() 返回最终 state
  → result["messages"] = 8 条消息
  → result["files"]    = {"mcp_request.txt": "..."}
```

## 🔑 4 个关键观察

### 1️⃣ 并行 tool_calls
第 1 轮一次发出 3 个 tool_call(`ls` + `write_file` + `web_search`),`ToolNode` **并发执行**。LLM 自动识别了"这三件事互不依赖"。

### 2️⃣ 文件 = 工作记忆
**Save → Search → Read-Back** 三阶段把文件当成 LLM 的"外脑":
- 早期把计划写下来 → 不被后续搜索结果冲淡
- 答题前读回来 → 注意力强制回归原始需求

### 3️⃣ Token 成本

| 轮次 | input | output | 累计 input |
|---|---|---|---|
| 1 | 1305 | 214 | 1305 |
| 2 | 1720 | 76 | 3025 |
| 3 | 1886 | 423 | 4911 |

**总 input = 4911,output = 713**。每轮 input 单调上涨(messages 不断追加),典型 ReAct 特征。

### 4️⃣ 退出条件
`tool_calls=[]` + `stop_reason='end_turn'` → 路由器走 `END` → invoke 返回。

## 📁 `result["files"]` 最终长啥样

```python
{
    "mcp_request.txt": (
        "User Request: Give me an overview of Model Context Protocol (MCP)\n\n"
        "Topics to cover:\n"
        "- What is MCP?\n"
        "- Purpose and goals\n"
        "- Key features\n"
        "- How it works\n"
        "- Use cases\n"
        "- Who developed it\n"
    )
}
```

**整次执行只产生 1 个文件**。`ls()` 调了但没用上(初始就是空),`web_search` 的结果**没落盘**(因为 mock 工具不写文件)—— 这跟下一节 4_full_agent 的 `tavily_search` 不同,后者会把搜索结果落盘。

## ✨ 一句话总结

<div style="background:#e7f7e9; padding:10px 24px; border-left:4px solid #28a745; border-radius:4px; width:97%;">

> 🔑 **`agent.invoke()` = 触发 LangGraph 状态机在 `agent ↔ tools` 之间反复跳**。
>
> 这次跳了 **3 轮**(LLM 调用 3 次,工具节点 2 次),按 `FILE_USAGE_INSTRUCTIONS` 教的 **"先 orient → 再 save → 再 search → 再 read-back → 最后 answer"** 走完,直到 LLM 给出一个无 `tool_calls` 的 AIMessage。
>
> **文件在这里的角色 ≠ 数据库,而是"控制 LLM 注意力的开关"** —— 把计划写下来,等需要时再读回 context 末尾,精准对抗 context rot。

</div>